# CTMC Particle Filtering Tutorial

### Import the CTMC SSM classes/objects and anything else required.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import xarray as xr

# CTMC SSM Objects
from ctmc_modules.ctmc_ssms import *

# Required imports from particles package
import particles
from particles import augmented_state_space_models as augssm
from particles.collectors import Moments

### Functions

In [ ]:
def sigmoid(x, a, b, m, k):
    """ result = a when t = 0. """
    result = a + (b-a) / (1 + np.exp(-k*(x-m)))
    result -= a + (b-a) / (1 + np.exp(k * m))
    result += a
    return result


def simulate_sigmoid_growth(mu0, max_growth, n, J, K, delta_t, sig_func_val=14):
    """ Simulate true_states and data where rates undergo Sigmoid growth. """
    
    L = mu0.shape[0]

    # k = 0
    true_states = [mu0.reshape(1, -1)]
    data = [np.sort(np.array([j % n for j in range(J)])).reshape(1, -1)]
    
    # k = 1, ..., K
    for k in range(1, K+1):
        # lams
        lams_k = np.array([sigmoid(k, mu0[i], mu0[i] + max_growth[i],
                                   K/2, sig_func_val / K)
                           for i in range(L)])
        true_states.append(lams_k.reshape(1, -1))
        
        # y
        P_mat = np.stack([compute_transition_prob_matrix(cur_lams, n, delta_t)
                          for cur_lams in true_states[-2]], axis=0)
        y_t = np.array([dists.Categorical(P_mat[np.arange(P_mat.shape[0]), y_i]).rvs()
                        for y_i in data[-1]])
        data.append(y_t)
    return true_states, data


def simulate_constant_rates(mu0, n, J, K, delta_t):
    """ Simulate true_states and data where rates remain constant. """
    
    # k = 0, ..., K
    true_states = [mu0.reshape(1, -1) for _ in k_series]
    
    # k = 0
    data = [ np.sort(np.array([j % n for j in range(N_rws)])).reshape(1, -1) ]

    # k = 1, ..., K
    for k in range(1, K+1):
        # y
        P_mat = np.stack([compute_transition_prob_matrix(cur_lams, n, delta_t)
                          for cur_lams in true_states[k-1]], axis=0)
        y_k = np.array([dists.Categorical(P_mat[np.arange(P_mat.shape[0]), y_i]).rvs()
                        for y_i in data[-1]])
        # y_k = np.array([dists.Categorical(P_mat[:, y_i]).rvs()
        #                 for y_i in data[-1]])
        data.append(y_k)
    return true_states, data
    

### Define parameters for SSM and particle filters

* `mu0[i]` and `var0[i]` is the mean and variance of the initial Gamma distribution for $\lambda_{pq}$ (for $\mathbb{P}(A_0)$), where $p$ and $q$ can be found using `lams_idx_to_gen_pos(i, n)`. The rate parameters are ordered as $\lambda^{0 \to 1},\, \lambda^{0 \to 2},\, \dots,\, \lambda^{0 \to n},\, \lambda^{1 \to 0},\, \lambda^{1 \to 2},\, \dots,\, \lambda^{(n-1) \to n}$. `mu0` and `var0` should be of shape `(n*(n-1),)`, where $n \cdot (n-1)$ is typically the total number of rate parameters in the CTMC.
* `delta_t`: $\Delta t$.
* `n`: the number of states the CTMC has.
* `J`: the number of random walkers traversing the CTMC.

In [ ]:
## CTMC SSM Parameters ##

delta_t = 0.02 # Time between observations. Keep small enough.
C = 1 # Scale the transition variance by C
n = 2 # Number of states in CTMC
mu0  = np.array([2, 3]) # For P(X_0)
var0 = np.array([0.2, 0.2]) # For P(X_0)
J = 100 # Number of random walkers traversing the CTMC

## Particle filtering, simulation and other parameters ##

N = 1000 # Number of particles used in particle filters

K = 300 # k = 0, 1, ..., K
k_series = np.arange(K + 1) # 0, 1, ..., K. Helpful for plotting.
time_points = delta_t * k_series # t_0=0, t_1, ..., t_K. Helpful for plotting.

### Create SSM object(s)

In [ ]:
## Gamma dist. params. for PX0 ##
a0, b0 = get_gamma_params_from_mean_var(mu0, var0)

## For a Bootstrap PF ##
ctmc_ssm = CTMC(a0=a0, b0=b0, n=n, J=J, C=C, delta_t=delta_t) 

### Simulate the data and true rates over time

`true_states` is a Python list of length $K+1$. `true_states[k]` is the true rates (lambdas) in a NumPy array of shape $(1, L)$ at $t_k$, where $L=n\cdot (n-1)$ is the number of rate parameters and $n$ is the number of states in the CTMC. To convert this to the generator at $t_k$, $A_k$, use `lams_to_gen(true_states[k].reshape(-1))`.

`data` is also a Python list of length $K+1$. `data[k]` is $y_k$, the states vector of the random walkers at $t_k$.

In [ ]:
## Sigmoid growth ##
max_growth = np.array([2, 1]) # Must be same shape as mu0
sigmoid_growth_param = 14 # Tune this to achieve desired shape of sigmoid growth
true_states, data = simulate_sigmoid_growth(mu0, max_growth, n, J, K, delta_t,
                                            sigmoid_growth_param)


## Store true lambdas in Pandas dataframe ##
lams_gen_positions = [lams_idx_to_gen_pos(i, n)
                      for i in range(true_states[0].shape[1])]
true_lams = pd.DataFrame(np.stack([true_state.reshape(-1)
                                   for true_state in true_states]),
                         columns=[f"λ_{p}{q}" for p, q in lams_gen_positions],
                         index=k_series)
true_lams = true_lams.rename_axis('k')

### Plot the true lambdas

In [ ]:
for col in true_lams.columns:
    plt.plot(k_series, true_lams[col], label=col)

plt.xlabel("k")
plt.ylabel("Value")
plt.title("True rates over time")
plt.legend()
plt.tight_layout()
plt.show()

### Plot data (if J not too large)

In [ ]:
if J <= 10:
    data_plot = np.vstack(data)
    
    fig, axes = plt.subplots(
        nrows=J,
        ncols=1,
        sharex=True,
        figsize=(8, 2.5 * n)
    )
    fig.suptitle("RW states over time", fontsize=14)
    
    # Ensure axes is always iterable (important if J == 1)
    if J == 1:
        axes = [axes]
    
    for j in range(J):
        axes[j].plot(k_series, data_plot[:, j])
        axes[j].set_ylabel(f"RW #{j+1}")
        axes[j].grid(True)
    
    axes[-1].set_xlabel("k")
    plt.tight_layout()
    plt.show()
else:
    print(f"Too many random walkers to plot: {J} RWs.")

### Run the Bootstrap Particle Filter

In [ ]:
## Run the bootstrap PF ##

fk_boot = augssm.AugmentedBootstrap(ssm=ctmc_ssm, data=data)
pf_boot = particles.SMC(fk=fk_boot, N=N, resampling='stratified', 
                        store_history=True, collect=[Moments()])
pf_boot.run()


## Store lambda particles and weights in xarray.Dataset ##

ds_boot = xr.Dataset({
    'X': xr.DataArray(
        np.stack([pf_boot.hist.X[k] for k in k_series]),
        dims=("k", "particle", "lam"),
        coords={
            "k": k_series,
            "lam": true_lams.columns.values,
        },
        name="Bootstrap PF Particles"
    ),
    'W': xr.DataArray(
        np.stack([pf_boot.hist.wgts[k].W for k in k_series]),
        dims=("k", "weight"),
        coords={
            "k": k_series
        },
        name="Bootstrap PF Weights"
    )
})


## Calculate quantiles and add into ds_boot ##

def weighted_quantile(values, weights, quantiles):
    """
    values: (particle,)
    weights: (particle,)
    quantiles: array-like in [0,1]
    """
    sorter = np.argsort(values)
    values = values[sorter]
    weights = weights[sorter]

    cdf = np.cumsum(weights)
    cdf = cdf / cdf[-1]

    return np.interp(quantiles, cdf, values)

qs = np.array([0.05, 0.5, 0.95]) # 95% interval

ds_boot["X_quantiles"] = xr.apply_ufunc(
    weighted_quantile,
    ds_boot["X"],
    ds_boot["W"],
    input_core_dims=[["particle"], ["weight"]],
    output_core_dims=[["quantile"]],
    vectorize=True,
    kwargs={"quantiles": qs},
    dask="parallelized",
    output_dtypes=[float],
).assign_coords(quantile=qs)

### Band plots (bad way): mean +/- 2 standard deviations

In [ ]:
means_boot =  np.stack([m['mean'] for m in pf_boot.summaries.moments])
vars_boot = np.stack([m['var'] for m in pf_boot.summaries.moments])

for lam_idx, lam in enumerate(true_lams.columns):
    plt.plot(k_series, true_lams[lam].values, label=f"True {lam}",
             color='red', alpha=0.7)
    plt.plot(means_boot[..., lam_idx], color="green",
             label="PF mean", alpha=0.7)
    plt.fill_between(k_series, 
                     y1=(means_boot[..., lam_idx]
                         -2*np.sqrt(vars_boot[..., lam_idx])), 
                     y2=(means_boot[..., lam_idx]
                         +2*np.sqrt(vars_boot[..., lam_idx])), 
                     color="green", alpha=0.3)
    plt.legend()
    plt.xlabel("k")
    plt.ylabel(f"Value of {lam}")
    p, q = lams_idx_to_gen_pos(lam_idx, n)
    plt.title(f"Boot PF band plot: $\\lambda^{{{p} \\to {q}}}$ | "
              + f"J={J}; N={N}; $\Delta t$={delta_t}; C={C}")
    plt.show()

### Band plots (better way): $5^{\text{th}}$, $50^{\text{th}}$, $95^{\text{th}}$ quantiles

In [ ]:
for lam_idx, lam in enumerate(true_lams.columns):
    median = ds_boot["X_quantiles"].sel(lam=lam, quantile=0.5)
    lq = ds_boot["X_quantiles"].sel(lam=lam, quantile=0.05)
    uq = ds_boot["X_quantiles"].sel(lam=lam, quantile=0.95)
    plt.plot(k_series, true_lams[lam].values, label=f"True {lam}",
             color='red', alpha=0.7)
    plt.plot(median, color="green",
             label="PF mean", alpha=0.7)
    plt.fill_between(k_series, 
                     y1=lq, 
                     y2=uq, 
                     color="green", alpha=0.3)
    plt.legend()
    plt.xlabel("k")
    plt.ylabel(f"Value of {lam}")
    p, q = lams_idx_to_gen_pos(lam_idx, n)
    plt.title(f"Boot PF band plot (quantiles): $\\lambda^{{{p} \\to {q}}}$ | "
              + f"J={J}; N={N}; $\Delta t$={delta_t}; C={C}")
    plt.show()

### Effective Sample Size (ESS) over time

In [ ]:
plt.plot(k_series, pf_boot.summaries.ESSs, color="red")
plt.xlabel("k")
plt.ylabel("ESS")
plt.title("ESS over time: Boot PF | "
          + f"J={J}; N={N}; $\Delta t$={delta_t}; C={C}")
plt.show()

### Choose some k value between 0 and K+1 (inclusive)

In [ ]:
k = np.random.randint(K+1)

### KDEs of filtering particles at k

Should use the weights.

In [ ]:
for lam in true_lams.columns:
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.kdeplot(x=ds_boot["X"].sel({'lam': lam, 'k': k}).values.reshape(-1),
                weights=ds_boot["W"].sel({'k': k}).values.reshape(-1),
                ax=ax, fill=True,
                color="skyblue", label="Boot")
    ax.axvline(x=true_lams.loc[k][lam], color='red', linestyle=':', linewidth=1.5, 
               label='True state')
    ax.set_xlabel("Value")
    ax.set_ylabel("Density")
    ax.set_title(f"Boot Filtering Dist @ k = {k}: {lam}")
    ax.legend()
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.show()

### Pairwise scatter plots at k

Plots do not currently use the weights.

In [ ]:
plot_df = (
    ds_boot["X"].sel(k=k)
    .to_pandas() # index: particle, columns: lambda
    .reset_index(drop=True)
)

sns.pairplot(
    plot_df,
    plot_kws={"alpha": 0.5, "s": 15}
)
plt.suptitle(f"Pairwise scatter at k = {k}: Boot", y=1.02)
plt.show()